In [ ]:
# Inisialisasi & Persiapan Data
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window
import pandas as pd

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("TugasMandiri5") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Membaca data transaksi dari HDFS[cite: 1]
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

# Menambahkan kolom pendapatan (unit_terjual * harga_satuan)[cite: 1]
df_transaksi = df_transaksi.withColumn(
    "pendapatan", 
    col("unit_terjual") * col("harga_satuan")
)

# 3. Membuat df_target dari dictionary data_target_cabang[cite: 1]
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("df_transaksi:")
df_transaksi.show(5)
print("df_target:")
df_target.show()

In [ ]:
#  Bagian A. Join & Perbandingan Target (DataFrame API)
# Meringkas total pendapatan per kota[cite: 1]
df_ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan menghitung pencapaian_persen[cite: 1]
df_bagian_a = df_ringkasan.join(df_target, on="kota", how="inner") \
    .withColumn(
        "pencapaian_persen",
        (col("total_pendapatan") / col("target_bulanan")) * 100
    ) \
    .orderBy(col("pencapaian_persen").desc())

df_bagian_a.show()

In [ ]:
# Bagian B. Window Function — Kategori Terlaris per Kota (DataFrame API)
# Agregasi total pendapatan per kota dan kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Menentukan Window berdasarkan kota[cite: 1]
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Mengambil kategori dengan pendapatan tertinggi per kota (top-1 dengan row_number)[cite: 1]
df_bagian_b = df_kategori_kota.withColumn("rn", row_number().over(window_kota)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy("kota")

df_bagian_b.show()

In [ ]:
#

#Bagian D: Kesimpulan (25%)

Berdasarkan analisis performa cabang, Purworejo (PIC: Fitri) menjadi cabang terbaik dengan pencapaian 127,3% (Rp38,19 juta dari target Rp30 juta), disokong oleh tingginya penjualan kategori Elektronik (Rp11,25 juta). Sebaliknya, Yogyakarta (PIC: Joko) menjadi cabang yang paling memerlukan perhatian khusus karena pencapaiannya terendah, yaitu hanya 49,8% (Rp29,87 juta dari target Rp60 juta). Manajemen direkomendasikan untuk menyesuaikan target Yogyakarta agar lebih realistis, meningkatkan promosi produk bernilai tinggi seperti Elektronik dan Fashion di Yogyakarta, serta mengadopsi strategi penjualan cabang Purworejo ke wilayah lain.